In [1]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = r"/server/my_model/my_trained_dictabert"

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)

model.eval()



def predict(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=256
    )

    with torch.no_grad():
        logits = model(**inputs).logits

    score = logits.view(-1)[0].item()

    return score



C:\Users\levm\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/server/my_model/my_trained_dictabert'. Use `repo_type` argument if needed.

In [31]:
import requests

session = requests.Session()

# 1. פותחים את האתר כדי לקבל cookies
session.get("https://www.scribens.com", verify=False)

# 2. שולחים request אמיתי
url = "https://www.scribens.fr/Scribens/OtherAlg_Ref_Servlet"

data = {
    "FunctionName": "Get_Correction",
    "Plugin": "Website_desktop",
    "Text": "השמש וכוכבי הלחת@ שמסובבים אותהח",
    "IdLanguage": "he",
    "IdLangDisplay": "he",
    "Tone": "nope",
    "Settings": "points:none|title:no|conclusion:no|inclusive:no|function:None"
}

headers = {
    "User-Agent": "Mozilla/5.0",
    "Origin": "https://www.scribens.com",
    "Referer": "https://www.scribens.com/",
}

res = session.post(url, data=data, headers=headers, verify=False)

print(res.text)

{"Map_Solutions":{},"ResultSt":"השמש וכוכבי הלכת שמסובבים אותה","NbRewriting":-1,"NbSummarizing":-1,"NbTranslation":-1,"DisplayPremiumPanel":false}


In [2]:
import pandas as pd
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =========================
# מודל
# =========================
model_path = r"/server/my_model/my_trained_dictabert"

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


# =========================
# פונקציית בניית טקסט
# =========================
def format_input(q, t, s):
    return f"[Q]{q}[T]{t}[S]{s}"


# =========================
# חילוץ שדות מתוך טקסט
# =========================
def extract_q_t_s(text):
    if not isinstance(text, str):
        return "", "", ""

    q = re.search(r"\[Q\]\s*(.*?)\s*\[T\]", text)
    t = re.search(r"\[T\]\s*(.*?)\s*\[S\]", text)
    s = re.search(r"\[S\]\s*(.*)", text)

    return (
        q.group(1).strip() if q else "",
        t.group(1).strip() if t else "",
        s.group(1).strip() if s else ""
    )


# =========================
# קריאת קובץ
# =========================
file_path = r"Y:\\שונות\\מלכי וציפי\\test_with_model_score_new_new.csv"

df = pd.read_csv(file_path, encoding="utf-8-sig", engine="python")

# חילוץ q,t,s
df[["q", "t", "s"]] = df["text"].apply(lambda x: pd.Series(extract_q_t_s(x)))

# סינון שורות ריקות
df = df[(df["q"] != "") & (df["t"] != "") & (df["s"] != "")].reset_index(drop=True)


# =========================
# יצירת טקסטים למודל
# =========================
texts = [format_input(r.q, r.t, r.s) for r in df.itertuples(index=False)]


# =========================
# אינפרנס בבאצ'ים
# =========================
BATCH_SIZE = 32
results = []

with torch.no_grad():
    for i in range(0, len(texts), BATCH_SIZE):
        batch_texts = texts[i:i + BATCH_SIZE]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=256
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        logits = model(**inputs).logits

        # התאמה לבינארי / מולטי-קלאס
        if logits.shape[1] == 1:
            scores = torch.sigmoid(logits).squeeze(-1)
        else:
            scores = torch.softmax(logits, dim=1)[:, 1]

        results.extend(scores.cpu().tolist())


# =========================
# הוספה ושמירה
# =========================
df["dictabert_score"] = results

output_path = file_path.replace(".csv", "_with_dictabert.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("נשמר:", output_path)

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/server/my_model/my_trained_dictabert'. Use `repo_type` argument if needed.

In [3]:
import os
print(os.getcwd())

C:\git\CleverCheck\server\services\try


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# התיקייה שבה נמצא המודל המאומן הטוב ביותר
model_path = "C:/git/CleverCheck/server/my_model/my_trained_dictabert"

# טעינת tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

# טעינת מודל מאומן
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)

# מצב inference
model.eval()


def format_natural_text(text):
    text = str(text)
    text = text.replace("[Q]", "שאלה:")
    text = text.replace("[T]", " תשובה נכונה:")
    text = text.replace("[S]", " תשובת תלמיד:")
    return text


def predict(text):
    text = format_natural_text(text)

    encoded = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )

    with torch.no_grad():
        output = model(**encoded)

    prediction = output.logits[0][0].item()

    return prediction




# דוגמה
text = """
[Q] מה גורם ליום ולילה?
[T] סיבוב כדור הארץ סביב עצמו
[S] כדור הארץ מסתובב סביב עצמו
"""

prediction = predict(text)

print("Predicted score:", prediction)

Predicted score: 0.6772354245185852


Predicted score: 0.04134140908718109


In [4]:

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_PATH = r"C:\git\CleverCheck/server/my_model/hebert_model_download"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

model.eval()  # חובה

# -------------------------
# פונקציית בניית קלט
# -------------------------
def build_input(question, reference, student):
    return f"שאלה: {question} תשובת מורה: {reference} תשובת תלמיד: {student}"


# -------------------------
# חיזוי
# -------------------------
def predict(question, reference, student):
    text = build_input(question, reference, student)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    # regression score
    return outputs.logits.item()


# -------------------------
# בדיקות לדוגמה
# -------------------------
tests = [
    {
        "question": "מה גורם ליום ולילה?",
        "reference": "סיבוב כדור הארץ סביב צירו.",
        "student": "סיבוב כדור הארץ סביב צירו.",
        "expected": "HIGH"
    }
]

print("START EVALUATION\n")

for t in tests:
    score = predict(t["question"], t["reference"], t["student"])

    print("Q:", t["question"])
    print("Student:", t["student"])
    print("Score:", round(score, 3))
    print("Expected:", t["expected"])
    print("-" * 50)

START EVALUATION

Q: מה גורם ליום ולילה?
Student: סיבוב כדור הארץ סביב צירו.
Score: 0.94
Expected: HIGH
--------------------------------------------------


In [ ]:
import pandas as pd
import re
file_path = r"H:\test_results_with_predictions.csv"

df = pd.read_csv(file_path, encoding="utf-8-sig", engine="python")

model2_scores = []
nkp_scores = []

for text in df["text"]:
    q = re.search(r"\[Q\]\s*(.*?)\s*\[T\]", text)
    t = re.search(r"\[T\]\s*(.*?)\s*\[S\]", text)
    s = re.search(r"\[S\]\s*(.*)", text)

    question = q.group(1).strip()
    teacher_answer = t.group(1).strip()
    student_answer = s.group(1).strip()

    model2_scores.append(
        get_model2_score(question, teacher_answer, student_answer)
    )

    nkp_scores.append(
        get_nkp_score(question, teacher_answer, student_answer)
    )

df["model2_score"] = model2_scores
df["nkp_score"] = nkp_scores
df.to_csv("output.csv", index=False, encoding="utf-8-sig")